In [8]:
%load_ext autoreload
%aimport helper, tests
%autoreload 1

In [1]:
import collections
import os
import helper as hp
import numpy as np
import project_tests as tests
from keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.models import Model
from keras.layers import GRU, Input, Dense, TimeDistributed, Activation, RepeatVector, Bidirectional
from keras.layers import Embedding
from keras.optimizers import Adam
from keras.losses import sparse_categorical_crossentropy

In [9]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 14275103635341590745
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 15208677376
locality {
  bus_id: 1
  links {
  }
}
incarnation: 4435037402477416334
physical_device_desc: "device: 0, name: Tesla V100-SXM2-16GB, pci bus id: 0000:00:04.0, compute capability: 7.0"
xla_global_id: 416903419
]
[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 13454143767982350331
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 15208677376
locality {
  bus_id: 1
  links {
  }
}
incarnation: 15696198334546321513
physical_device_desc: "device: 0, name: Tesla V100-SXM2-16GB, pci bus id: 0000:00:04.0, compute capability: 7.0"
xla_global_id: 416903419
]


In [10]:
def load_data(path):
    """
    Load dataset
    """
    input_file = os.path.join(path)
    with open(input_file, "r") as f:
        data = f.read()

    return data.split('\n')
# Load English data
english_sentences = load_data('small_vocab_en.txt')
# Load French data
french_sentences = load_data('small_vocab_fr.txt')

print('Dataset Loaded')

Dataset Loaded
Dataset Loaded


In [11]:
#text has already been converted to lowercase, mostly preprocessed
for i in range(5):
  print("Line", i, ": ", english_sentences[i], french_sentences[i])

Line 0 :  new jersey is sometimes quiet during autumn , and it is snowy in april . new jersey est parfois calme pendant l' automne , et il est neigeux en avril .
Line 1 :  the united states is usually chilly during july , and it is usually freezing in november . les états-unis est généralement froid en juillet , et il gèle habituellement en novembre .
Line 2 :  california is usually quiet during march , and it is usually hot in june . california est généralement calme en mars , et il est généralement chaud en juin .
Line 3 :  the united states is sometimes mild during june , and it is cold in september . les états-unis est parfois légère en juin , et il fait froid en septembre .
Line 4 :  your least liked fruit is the grape , but my least liked is the apple . votre moins aimé fruit est le raisin , mais mon moins aimé est la pomme .
Line 0 :  new jersey is sometimes quiet during autumn , and it is snowy in april . new jersey est parfois calme pendant l' automne , et il est neigeux en av

In [12]:
#find unique words
english_counter = collections.Counter([word for sentence in english_sentences for word in sentence.split()])
french_counter = collections.Counter([word for sentence in french_sentences for word in sentence.split()])

In [13]:
print("total number of English words: ", len([word for sentence in english_sentences for word in sentence.split()]))
print("Unique English words: ", len(english_counter))
print("10 most common English words: ")
print('"' + '" "'.join(list(zip(*english_counter.most_common(10)))[0]) + '"')
print("total number of French words: ", len([word for sentence in french_sentences for word in sentence.split()]))
print("Unique French words: ", len(french_counter))
print("10 most common French words: ")
print('"' + '" "'.join(list(zip(*french_counter.most_common(10)))[0]) + '"')

total number of English words:  1823250
Unique English words:  227
10 most common English words: 
"is" "," "." "in" "it" "during" "the" "but" "and" "sometimes"
total number of French words:  1961295
Unique French words:  355
10 most common French words: 
"est" "." "," "en" "il" "les" "mais" "et" "la" "parfois"
total number of English words:  1823250
Unique English words:  227
10 most common English words: 
"is" "," "." "in" "it" "during" "the" "but" "and" "sometimes"
total number of French words:  1961295
Unique French words:  355
10 most common French words: 
"est" "." "," "en" "il" "les" "mais" "et" "la" "parfois"


In [7]:

#tokenize and pad
#using Keras' tokenizer function
def tokenize(x):
    """
    Tokenize x
    :param x: List of sentences/strings to be tokenized
    :return: Tuple of (tokenized x data, tokenizer used to tokenize x)
    """
    x_tk = Tokenizer(char_level = False)
    x_tk.fit_on_texts(x)
    return x_tk.texts_to_sequences(x), x_tk
#english tokenized has the sentences but replaces words with numbers, english_tokenizer has each unique word with its token
english_tokenized, english_tokenizer = tokenize(english_sentences)
french_tokenized, french_tokenizer = tokenize(french_sentences)


In [14]:
#print(english_tokenizer.word_index)
print(english_tokenized[:10])

[[17, 23, 1, 8, 67, 4, 39, 7, 3, 1, 55, 2, 44], [5, 20, 21, 1, 9, 62, 4, 43, 7, 3, 1, 9, 51, 2, 45], [22, 1, 9, 67, 4, 38, 7, 3, 1, 9, 68, 2, 34], [5, 20, 21, 1, 8, 64, 4, 34, 7, 3, 1, 57, 2, 42], [29, 12, 16, 13, 1, 5, 82, 6, 30, 12, 16, 1, 5, 83], [31, 11, 13, 1, 5, 84, 6, 30, 11, 1, 5, 82], [18, 1, 66, 4, 47, 6, 3, 1, 9, 62, 2, 43], [17, 23, 1, 60, 4, 35, 7, 3, 1, 10, 68, 2, 38], [49, 12, 16, 13, 1, 5, 85, 6, 30, 12, 16, 1, 5, 82], [5, 20, 21, 1, 8, 60, 4, 36, 7, 3, 1, 8, 56, 2, 45]]


In [15]:
for i, (sent, token_sent) in enumerate(zip(english_sentences[:10], english_tokenized[:10])):
  print('Sentence: ', i+1)
  print('Input: ', sent)
  print('Output: ', token_sent)

Sentence:  1
Input:  new jersey is sometimes quiet during autumn , and it is snowy in april .
Output:  [17, 23, 1, 8, 67, 4, 39, 7, 3, 1, 55, 2, 44]
Sentence:  2
Input:  the united states is usually chilly during july , and it is usually freezing in november .
Output:  [5, 20, 21, 1, 9, 62, 4, 43, 7, 3, 1, 9, 51, 2, 45]
Sentence:  3
Input:  california is usually quiet during march , and it is usually hot in june .
Output:  [22, 1, 9, 67, 4, 38, 7, 3, 1, 9, 68, 2, 34]
Sentence:  4
Input:  the united states is sometimes mild during june , and it is cold in september .
Output:  [5, 20, 21, 1, 8, 64, 4, 34, 7, 3, 1, 57, 2, 42]
Sentence:  5
Input:  your least liked fruit is the grape , but my least liked is the apple .
Output:  [29, 12, 16, 13, 1, 5, 82, 6, 30, 12, 16, 1, 5, 83]
Sentence:  6
Input:  his favorite fruit is the orange , but my favorite is the grape .
Output:  [31, 11, 13, 1, 5, 84, 6, 30, 11, 1, 5, 82]
Sentence:  7
Input:  paris is relaxing during december , but it is usually 

In [17]:
#pad sentences
def pad(x, length=None):
    """
    Pad x
    :param x: List of sequences.
    :param length: Length to pad the sequence to.  If None, use length of longest sequence in x.
    :return: Padded numpy array of sequences
    """
    # TODO: Implement
    if length is None:
        length = max([len(sentence) for sentence in x])
    return pad_sequences(x, maxlen = length, padding = 'post')
#tests.test_pad(pad)
english_pad = pad(english_tokenized)
french_pad = pad(french_tokenized)

In [18]:
for i, (token_sent, pad_sent) in enumerate(zip(english_tokenized[:10], english_pad[:10])):
    print('Sequence {} in x'.format(i + 1))
    print('  Input:  {}'.format(np.array(token_sent)))
    print('  Output: {}'.format(pad_sent))
for i, (token_sent, pad_sent) in enumerate(zip(french_tokenized[:10], french_pad[:10])):
    print('Sequence {} in x'.format(i + 1))
    print('  Input:  {}'.format(np.array(token_sent)))
    print('  Output: {}'.format(pad_sent))

Sequence 1 in x
  Input:  [17 23  1  8 67  4 39  7  3  1 55  2 44]
  Output: [17 23  1  8 67  4 39  7  3  1 55  2 44  0  0]
Sequence 2 in x
  Input:  [ 5 20 21  1  9 62  4 43  7  3  1  9 51  2 45]
  Output: [ 5 20 21  1  9 62  4 43  7  3  1  9 51  2 45]
Sequence 3 in x
  Input:  [22  1  9 67  4 38  7  3  1  9 68  2 34]
  Output: [22  1  9 67  4 38  7  3  1  9 68  2 34  0  0]
Sequence 4 in x
  Input:  [ 5 20 21  1  8 64  4 34  7  3  1 57  2 42]
  Output: [ 5 20 21  1  8 64  4 34  7  3  1 57  2 42  0]
Sequence 5 in x
  Input:  [29 12 16 13  1  5 82  6 30 12 16  1  5 83]
  Output: [29 12 16 13  1  5 82  6 30 12 16  1  5 83  0]
Sequence 6 in x
  Input:  [31 11 13  1  5 84  6 30 11  1  5 82]
  Output: [31 11 13  1  5 84  6 30 11  1  5 82  0  0  0]
Sequence 7 in x
  Input:  [18  1 66  4 47  6  3  1  9 62  2 43]
  Output: [18  1 66  4 47  6  3  1  9 62  2 43  0  0  0]
Sequence 8 in x
  Input:  [17 23  1 60  4 35  7  3  1 10 68  2 38]
  Output: [17 23  1 60  4 35  7  3  1 10 68  2 38  0  0]
Se

In [19]:
def preprocess(x, y):
    """
    Preprocess x and y
    :param x: Feature List of sentences
    :param y: Label List of sentences
    :return: Tuple of (Preprocessed x, Preprocessed y, x tokenizer, y tokenizer)
    """
    preprocess_x, x_tk = tokenize(x)
    preprocess_y, y_tk = tokenize(y)

    preprocess_x = pad(preprocess_x)
    preprocess_y = pad(preprocess_y)

    # Keras's sparse_categorical_crossentropy function requires the labels to be in 3 dimensions
    preprocess_y = preprocess_y.reshape(*preprocess_y.shape, 1)

    return preprocess_x, preprocess_y, x_tk, y_tk
preproc_english_sentences, preproc_french_sentences, english_tokenizer, french_tokenizer =\
    preprocess(english_sentences, french_sentences)

max_english_sequence_length = preproc_english_sentences.shape[1]
max_french_sequence_length = preproc_french_sentences.shape[1]
english_vocab_size = len(english_tokenizer.word_index)
french_vocab_size = len(french_tokenizer.word_index)

print('Data Preprocessed')
print("Max English sentence length:", max_english_sequence_length)
print("Max French sentence length:", max_french_sequence_length)
print("English vocabulary size:", english_vocab_size)
print("French vocabulary size:", french_vocab_size)

Data Preprocessed
Max English sentence length: 15
Max French sentence length: 21
English vocabulary size: 199
French vocabulary size: 344


In [20]:
#indexes back to translated words
def logits_to_text(logits, tokenizer):
    """
    Turn logits from a neural network into text using the tokenizer
    :param logits: Logits from a neural network
    :param tokenizer: Keras Tokenizer fit on the labels
    :return: String that represents the text of the logits
    """
    index_to_words = {id: word for word, id in tokenizer.word_index.items()}
    index_to_words[0] = '<PAD>'

    return ' '.join([index_to_words[prediction] for prediction in np.argmax(logits, 1)])

print('`logits_to_text` function loaded.')

`logits_to_text` function loaded.


In [21]:
import numpy as np
from keras.losses import sparse_categorical_crossentropy
from keras.models import Sequential
from keras.preprocessing.text import Tokenizer
from keras.utils import to_categorical
def _test_model(model, input_shape, output_sequence_length, french_vocab_size):
    if isinstance(model, Sequential):
        model = model.model

    assert model.input_shape == (None, *input_shape[1:]),\
        'Wrong input shape. Found input shape {} using parameter input_shape={}'.format(model.input_shape, input_shape)

    assert model.output_shape == (None, output_sequence_length, french_vocab_size),\
        'Wrong output shape. Found output shape {} using parameters output_sequence_length={} and french_vocab_size={}'\
            .format(model.output_shape, output_sequence_length, french_vocab_size)

    assert model.loss is not None,\
        'No loss function set.  Apply the `compile` function to the model.'

    assert str(model.compiled_loss) == str(sparse_categorical_crossentropy),\
        'Not using "sparse_categorical_crossentropy" as a loss function'
def test_tokenize(tokenize):
    sentences = [
        'The quick brown fox jumps over the lazy dog .',
        'By Jove , my quick study of lexicography won a prize .',
        'This is a short sentence .']
    tokenized_sentences, tokenizer = tokenize(sentences)
    assert tokenized_sentences == tokenizer.texts_to_sequences(sentences),\
        'Tokenizer returned and doesn\'t generate the same sentences as the tokenized sentences returned. '

def test_pad(pad):
    tokens = [
        [i for i in range(4)],
        [i for i in range(6)],
        [i for i in range(3)]]
    padded_tokens = pad(tokens)
    padding_id = padded_tokens[0][-1]
    true_padded_tokens = np.array([
        [i for i in range(4)] + [padding_id]*2,
        [i for i in range(6)],
        [i for i in range(3)] + [padding_id]*3])
    assert isinstance(padded_tokens, np.ndarray),\
        'Pad returned the wrong type.  Found {} type, expected numpy array type.'
    assert np.all(padded_tokens == true_padded_tokens), 'Pad returned the wrong results.'

    padded_tokens_using_length = pad(tokens, 9)
    assert np.all(padded_tokens_using_length == np.concatenate((true_padded_tokens, np.full((3, 3), padding_id)), axis=1)),\
        'Using length argument return incorrect results'


def test_simple_model(simple_model):
    input_shape = (137861, 21, 1)
    output_sequence_length = 21
    english_vocab_size = 199
    french_vocab_size = 344

    model = simple_model(input_shape, output_sequence_length, english_vocab_size, french_vocab_size)
    _test_model(model, input_shape, output_sequence_length, french_vocab_size)

In [22]:
#simple RNN to translate English to French

def RNN_model(input_shape, output_sequence_length, english_vocab_size, french_vocab_size):
    """
    :param input_shape: Tuple of input shape
    :param output_sequence_length: Length of output sequence
    :param english_vocab_size: Number of unique English words in the dataset
    :param french_vocab_size: Number of unique French words in the dataset
    :return: Keras model built, but not trained
    """
    learning_rate = 1e-3
    input_seq = Input(input_shape[1:])
    rnn = GRU(64, return_sequences=True)(input_seq)
    logits = TimeDistributed(Dense(french_vocab_size))(rnn)
    model = Model(input_seq, Activation('softmax')(logits))
    model.compile(loss=sparse_categorical_crossentropy, optimizer=Adam(learning_rate), metrics = ['accuracy'])

    return model

#test_simple_model(RNN_model)

#reshape inputs for RNN
tmp_x = pad(preproc_english_sentences, max_french_sequence_length)
tmp_x = tmp_x.reshape((-1, preproc_french_sentences.shape[-2], 1))

simple_rnn_model = RNN_model(tmp_x.shape, max_french_sequence_length, english_vocab_size, french_vocab_size)
simple_rnn_model.fit(tmp_x, preproc_french_sentences, batch_size = 1024, epochs=10, validation_split=0.2)

# Print prediction(s)
print(logits_to_text(simple_rnn_model.predict(tmp_x[:1])[0], french_tokenizer))



Epoch 1/10
108/108 [==============================] - 7s 14ms/step - loss: 3.4220 - accuracy: 0.4155 - val_loss: nan - val_accuracy: 0.4568
Epoch 2/10
108/108 [==============================] - 1s 8ms/step - loss: 2.5041 - accuracy: 0.4659 - val_loss: nan - val_accuracy: 0.4733
Epoch 3/10
108/108 [==============================] - 1s 7ms/step - loss: 2.2951 - accuracy: 0.4918 - val_loss: nan - val_accuracy: 0.5133
Epoch 4/10
108/108 [==============================] - 1s 7ms/step - loss: 2.0495 - accuracy: 0.5349 - val_loss: nan - val_accuracy: 0.5582
Epoch 5/10
108/108 [==============================] - 1s 7ms/step - loss: 1.8511 - accuracy: 0.5650 - val_loss: nan - val_accuracy: 0.5770
Epoch 6/10
108/108 [==============================] - 1s 7ms/step - loss: 1.7305 - accuracy: 0.5761 - val_loss: nan - val_accuracy: 0.5776
Epoch 7/10
108/108 [==============================] - 1s 7ms/step - loss: 1.6478 - accuracy: 0.5797 - val_loss: nan - val_accuracy: 0.5826
Epoch 8/10
108/108 [======

In [23]:
#Better model. Add Embeddings, not just

from keras.models import Sequential
def embed_model(input_shape, output_sequence_length, english_vocab_size, french_vocab_size):
    """
    Build and train a RNN model using word embedding on x and y
    :param input_shape: Tuple of input shape
    :param output_sequence_length: Length of output sequence
    :param english_vocab_size: Number of unique English words in the dataset
    :param french_vocab_size: Number of unique French words in the dataset
    :return: Keras model built, but not trained
    """
    learning_rate = 1e-3
    rnn = GRU(64, return_sequences=True, activation="tanh")
    embedding = Embedding(french_vocab_size, 64, input_length = input_shape[1])
    logits = TimeDistributed(Dense(french_vocab_size, activation="softmax"))
    model = Sequential()
    #add the layers
    model.add(embedding)
    model.add(rnn)
    model.add(logits)
    model.compile(loss=sparse_categorical_crossentropy, optimizer=Adam(learning_rate), metrics = ['accuracy'])
    return model

tmp_x = pad(preproc_english_sentences, max_french_sequence_length)
tmp_x = tmp_x.reshape((-1, preproc_french_sentences.shape[-2]))

embeded_model = embed_model(
    tmp_x.shape,
    max_french_sequence_length,
    english_vocab_size,
    french_vocab_size)

embeded_model.fit(tmp_x, preproc_french_sentences, batch_size=1024, epochs=10, validation_split=0.2)


# TODO: Print prediction(s)
print(logits_to_text(embeded_model.predict(tmp_x[:1])[0], french_tokenizer))






Epoch 1/10
108/108 [==============================] - 3s 13ms/step - loss: 3.7668 - accuracy: 0.4010 - val_loss: nan - val_accuracy: 0.4093
Epoch 2/10
108/108 [==============================] - 1s 9ms/step - loss: 2.6558 - accuracy: 0.4515 - val_loss: nan - val_accuracy: 0.5151
Epoch 3/10
108/108 [==============================] - 1s 9ms/step - loss: 1.9516 - accuracy: 0.5620 - val_loss: nan - val_accuracy: 0.6070
Epoch 4/10
108/108 [==============================] - 1s 9ms/step - loss: 1.4635 - accuracy: 0.6403 - val_loss: nan - val_accuracy: 0.6798
Epoch 5/10
108/108 [==============================] - 1s 10ms/step - loss: 1.1360 - accuracy: 0.7210 - val_loss: nan - val_accuracy: 0.7555
Epoch 6/10
108/108 [==============================] - 2s 17ms/step - loss: 0.9100 - accuracy: 0.7713 - val_loss: nan - val_accuracy: 0.7859
Epoch 7/10
108/108 [==============================] - 1s 12ms/step - loss: 0.7667 - accuracy: 0.7955 - val_loss: nan - val_accuracy: 0.8068
Epoch 8/10
108/108 [===

In [24]:
#Bidirectional RNNs Model

def bd_model(input_shape, output_sequence_length, english_vocab_size, french_vocab_size):
    """
    Build and train a bidirectional RNN model on x and y
    :param input_shape: Tuple of input shape
    :param output_sequence_length: Length of output sequence
    :param english_vocab_size: Number of unique English words in the dataset
    :param french_vocab_size: Number of unique French words in the dataset
    :return: Keras model built, but not trained
    """
    learning_rate = 1e-3
    model = Sequential()
    model.add(Bidirectional(GRU(128, return_sequences=True, dropout=0.1), input_shape = input_shape[1:]))
    model.add(TimeDistributed(Dense(french_vocab_size, activation="softmax")))
    model.compile(loss = sparse_categorical_crossentropy,
                 optimizer = Adam(learning_rate),
                 metrics = ['accuracy'])
    return model

tmp_x = pad(preproc_english_sentences, preproc_french_sentences.shape[1])
tmp_x = tmp_x.reshape((-1, preproc_french_sentences.shape[-2], 1))
bidi_model = bd_model(
    tmp_x.shape,
    preproc_french_sentences.shape[1],
    len(english_tokenizer.word_index)+1,
    len(french_tokenizer.word_index)+1)


bidi_model.fit(tmp_x, preproc_french_sentences, batch_size=1024, epochs=20, validation_split=0.2)

# Print prediction(s)
print(logits_to_text(bidi_model.predict(tmp_x[:1])[0], french_tokenizer))

Epoch 1/20
108/108 [==============================] - 6s 21ms/step - loss: 2.6870 - accuracy: 0.4974 - val_loss: 1.8324 - val_accuracy: 0.5661
Epoch 2/20
108/108 [==============================] - 2s 15ms/step - loss: 1.7052 - accuracy: 0.5770 - val_loss: 1.5045 - val_accuracy: 0.6064
Epoch 3/20
108/108 [==============================] - 1s 13ms/step - loss: 1.4817 - accuracy: 0.6069 - val_loss: 1.3845 - val_accuracy: 0.6190
Epoch 4/20
108/108 [==============================] - 2s 15ms/step - loss: 1.3681 - accuracy: 0.6242 - val_loss: 1.3239 - val_accuracy: 0.6262
Epoch 5/20
108/108 [==============================] - 2s 16ms/step - loss: 1.2934 - accuracy: 0.6381 - val_loss: 1.2992 - val_accuracy: 0.6239
Epoch 6/20
108/108 [==============================] - 2s 20ms/step - loss: 1.2391 - accuracy: 0.6473 - val_loss: 1.2850 - val_accuracy: 0.6240
Epoch 7/20
108/108 [==============================] - 2s 17ms/step - loss: 1.1961 - accuracy: 0.6551 - val_loss: 1.2895 - val_accuracy: 0.6233

In [26]:
#ENCODER - Decoder model

def encdec_model(input_shape, output_sequence_length, english_vocab_size, french_vocab_size):
    """
    Build and train an encoder-decoder model on x and y
    :param input_shape: Tuple of input shape
    :param output_sequence_length: Length of output sequence
    :param english_vocab_size: Number of unique English words in the dataset
    :param french_vocab_size: Number of unique French words in the dataset
    :return: Keras model built, but not trained
    """
    # OPTIONAL: Implement
    learning_rate = 1e-3
    model = Sequential()
    model.add(GRU(128, input_shape = input_shape[1:], return_sequences = False))
    model.add(RepeatVector(output_sequence_length))
    model.add(GRU(128, return_sequences = True))
    model.add(TimeDistributed(Dense(french_vocab_size, activation = 'softmax')))

    model.compile(loss = sparse_categorical_crossentropy,
                 optimizer = Adam(learning_rate),
                 metrics = ['accuracy'])
    return model



# OPTIONAL: Train and Print prediction(s)
tmp_x = pad(preproc_english_sentences)
tmp_x = tmp_x.reshape((-1, preproc_english_sentences.shape[1], 1))

encodeco_model = encdec_model(
    tmp_x.shape,
    preproc_french_sentences.shape[1],
    len(english_tokenizer.word_index)+1,
    len(french_tokenizer.word_index)+1)

encodeco_model.fit(tmp_x, preproc_french_sentences, batch_size=1024, epochs=20, validation_split=0.2)

print(logits_to_text(encodeco_model.predict(tmp_x[:1])[0], french_tokenizer))

Epoch 1/20
108/108 [==============================] - 6s 19ms/step - loss: 3.0047 - accuracy: 0.4371 - val_loss: 2.4759 - val_accuracy: 0.4850
Epoch 2/20
108/108 [==============================] - 1s 12ms/step - loss: 2.3312 - accuracy: 0.4964 - val_loss: 2.2185 - val_accuracy: 0.5025
Epoch 3/20
108/108 [==============================] - 1s 12ms/step - loss: 2.1150 - accuracy: 0.5159 - val_loss: 1.9772 - val_accuracy: 0.5522
Epoch 4/20
108/108 [==============================] - 1s 12ms/step - loss: 1.8504 - accuracy: 0.5553 - val_loss: 1.7488 - val_accuracy: 0.5662
Epoch 5/20
108/108 [==============================] - 1s 12ms/step - loss: 1.6894 - accuracy: 0.5728 - val_loss: 1.6368 - val_accuracy: 0.5765
Epoch 6/20
108/108 [==============================] - 2s 14ms/step - loss: 1.5918 - accuracy: 0.5837 - val_loss: 1.5506 - val_accuracy: 0.5902
Epoch 7/20
108/108 [==============================] - 2s 14ms/step - loss: 1.5144 - accuracy: 0.5952 - val_loss: 1.4794 - val_accuracy: 0.5995

In [27]:
def model_final(input_shape, output_sequence_length, english_vocab_size, french_vocab_size):
    """
    Build and train a model that incorporates embedding, encoder-decoder, and bidirectional RNN on x and y
    :param input_shape: Tuple of input shape
    :param output_sequence_length: Length of output sequence
    :param english_vocab_size: Number of unique English words in the dataset
    :param french_vocab_size: Number of unique French words in the dataset
    :return: Keras model built, but not trained
    """
    # TODO: Implement
    model = Sequential()
    model.add(Embedding(input_dim=english_vocab_size,output_dim=128,input_length=input_shape[1]))
    model.add(Bidirectional(GRU(256,return_sequences=False)))
    model.add(RepeatVector(output_sequence_length))
    model.add(Bidirectional(GRU(256,return_sequences=True)))
    model.add(TimeDistributed(Dense(french_vocab_size,activation='softmax')))
    learning_rate = 0.005

    model.compile(loss = sparse_categorical_crossentropy,
                 optimizer = Adam(learning_rate),
                 metrics = ['accuracy'])

    return model



print('Final Model Loaded')
# TODO: Train the final model

Final Model Loaded


In [30]:
def final_predictions(x, y, x_tk, y_tk):
    """
    Gets predictions using the final model
    :param x: Preprocessed English data
    :param y: Preprocessed French data
    :param x_tk: English tokenizer
    :param y_tk: French tokenizer
    """
    # TODO: Train neural network using model_final
    tmp_X = pad(preproc_english_sentences)
    model = model_final(tmp_X.shape,
                        preproc_french_sentences.shape[1],
                        len(english_tokenizer.word_index)+1,
                        len(french_tokenizer.word_index)+1)

    model.fit(tmp_X, preproc_french_sentences, batch_size = 1024, epochs = 17, validation_split = 0.2)

    ## DON'T EDIT ANYTHING BELOW THIS LINE
    y_id_to_word = {value: key for key, value in y_tk.word_index.items()}
    y_id_to_word[0] = '<PAD>'

    sentence = 'he saw a old yellow truck'
    sentence = [x_tk.word_index[word] for word in sentence.split()]
    sentence = pad_sequences([sentence], maxlen=x.shape[-1], padding='post')
    sentences = np.array([sentence[0], x[0]])
    predictions = model.predict(sentences, len(sentences))

    print('Sample 1:')
    print(' '.join([y_id_to_word[np.argmax(x)] for x in predictions[0]]))
    print('Il a vu un vieux camion jaune')
    print('Sample 2:')
    print(' '.join([y_id_to_word[np.argmax(x)] for x in predictions[1]]))
    print(' '.join([y_id_to_word[np.max(x)] for x in y[0]]))


final_predictions(preproc_english_sentences, preproc_french_sentences, english_tokenizer, french_tokenizer)

Epoch 1/17
108/108 [==============================] - 11s 50ms/step - loss: 2.1142 - accuracy: 0.5174 - val_loss: 1.4346 - val_accuracy: 0.6187
Epoch 2/17
108/108 [==============================] - 4s 41ms/step - loss: 1.0762 - accuracy: 0.6992 - val_loss: 0.9425 - val_accuracy: 0.7195
Epoch 3/17
108/108 [==============================] - 4s 37ms/step - loss: 0.6639 - accuracy: 0.8010 - val_loss: 0.5081 - val_accuracy: 0.8426
Epoch 4/17
108/108 [==============================] - 4s 37ms/step - loss: 0.3831 - accuracy: 0.8848 - val_loss: 0.2722 - val_accuracy: 0.9208
Epoch 5/17
108/108 [==============================] - 4s 38ms/step - loss: 0.2232 - accuracy: 0.9341 - val_loss: 0.1914 - val_accuracy: 0.9442
Epoch 6/17
108/108 [==============================] - 4s 37ms/step - loss: 0.1636 - accuracy: 0.9515 - val_loss: 0.1467 - val_accuracy: 0.9569
Epoch 7/17
108/108 [==============================] - 4s 37ms/step - loss: 0.1198 - accuracy: 0.9643 - val_loss: 0.1265 - val_accuracy: 0.962

1/1 [==============================] - 1s 1s/step
Sample 1:
il a vu un vieux camion jaune <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD>
Il a vu un vieux camion jaune
Sample 2:
new jersey est parfois calme pendant l' automne et il est en avril avril <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD>
new jersey est parfois calme pendant l' automne et il est neigeux en avril <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD>
